# 12 — Word2Vec with Gensim

**Learning objective.** Train a tiny embedding model with a standard NLP library and inspect learned similarities.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## 🧠 Visual engineering mental model

![Causal mindmap](assets/mindmaps/12_word2vec_gensim.svg)

Follow the information flow, then ask what the control knob changes before touching code.

## 🎛️ Change map — cause → representation → behavior

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase **window size** | more distant context pairs are trained | embeddings become more topical |
| Switch CBOW ↔ Skip-gram | prediction direction changes | rare-word behavior and training dynamics change |
| Increase epochs/dimension | model fits corpus more strongly | small corpora can overfit/noisy geometry |

**Engineering habit:** make one intervention, state the expected direction of change, then measure it.

## 🔮 Predict before you run

1. Why can two words become close even if they never appear adjacent?
2. What happens to a static embedding when one word has multiple senses?

### When to use
Useful for compact static lexical representations and for understanding predictive embedding learning.

### When not / caution
Not enough when the same word must change meaning with context.

### Debugging lens
Check corpus frequency, window pairs and random seed before interpreting a single nearest-neighbor result.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from gensim.models import Word2Vec
sentences=[
 ['king','royal','man'],['queen','royal','woman'],['prince','royal','man'],['princess','royal','woman'],
 ['man','human','adult'],['woman','human','adult'],['king','queen','palace'],['prince','princess','palace']
]*40
model=Word2Vec(sentences=sentences,vector_size=16,window=2,min_count=1,sg=1,workers=1,epochs=80,seed=42)
for w in ['king','queen','man','woman']:
    print(w, model.wv.most_similar(w,topn=3))

king [('princess', 0.9708093404769897), ('palace', 0.9648391604423523), ('prince', 0.952725350856781)]
queen [('palace', 0.9835293889045715), ('prince', 0.9784473180770874), ('adult', 0.9645273089408875)]
man [('palace', 0.9766647219657898), ('royal', 0.9699306488037109), ('princess', 0.965228259563446)]
woman [('palace', 0.9741625785827637), ('royal', 0.9698006510734558), ('prince', 0.9653858542442322)]


In [3]:
print('vector size:',model.wv['king'].shape)
print('cosine(king, queen):',round(model.wv.similarity('king','queen'),3))
print('cosine(king, woman):',round(model.wv.similarity('king','woman'),3))

vector size: (16,)
cosine(king, queen): 0.949
cosine(king, woman): 0.941


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Explain what Word2Vec optimizes conceptually
- Use cosine similarity to inspect embedding geometry